# **Access Drive File**

# **A Dual-View Semantic Fusion Network Model for Enhanced Just-in-Time Software Defect Prediction**

# **libraries**

In [ ]:
pip install torch-geometric

# **Commit Code**

# **Z-score Normalization**

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1️⃣ Load input CSV file
# -----------------------------
input_csv = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/Numerical/jit_Numerical Data.csv"
output_csv = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/zscore_output.csv"

df = pd.read_csv(input_csv)

# -----------------------------
# 2️⃣ Select numeric columns safely
#    EXCLUDING defect_label and any ID/URL/strings
# -----------------------------
exclude_cols = ['bug.id', 'revision.id.buggy', 'revision.id.fixed', 'report.id', 'report.url', 'commit_message', 'defect_label']
numeric_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]


print("numeric_cols :::::: ", numeric_cols)
# # -----------------------------
# # 3️⃣ Apply Z-score normalization only where std != 0
# -----------------------------
for col in numeric_cols:
    if df[col].std() != 0:
        df[col] = (df[col] - df[col].mean()) / df[col].std()
    else:
        df[col] = 0  # If std=0, set normalized value to 0

# -----------------------------
# 4️⃣ Save normalized CSV
# -----------------------------
df.to_csv(output_csv, index=False)

print("✅ Z-score normalization completed safely.")
print("🛑 defect_label and ID/string columns kept unchanged.")
print(f"📁 Output saved as: {output_csv}")
print(df.head())


# **Handle numeric missing values**

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1️⃣ Load dataset
# -----------------------------
input_csv = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/zscore_output.csv"
output_csv = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/handled_missing.csv"

df = pd.read_csv(input_csv)

# -----------------------------
# 2️⃣ Separate label column
# -----------------------------
label_col = 'defect_label'

# -----------------------------
# 3️⃣ Handle numeric missing values
# -----------------------------
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
numeric_cols = numeric_cols.drop(label_col, errors='ignore')


# Strategy:
# - history-related metrics → fill with 0
# - others → fill with median
for col in numeric_cols:
    if 'hist' in col.lower() or 'prev' in col.lower() or 'count' in col.lower():
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna(df[col].median())

# -----------------------------
# 4️⃣ Handle categorical columns (if any)
# -----------------------------
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# -----------------------------
# 5️⃣ Save cleaned dataset
# -----------------------------
df.to_csv(output_csv, index=False)

print("✅ Missing values handled successfully")
print("🛑 defect_label left unchanged")
print(f"📁 Output saved as: {output_csv}")
df

# **Label Processing, Imbalance Handling**

In [ ]:
# ===============================
# Defect Dataset: Label Processing, Imbalance Handling & Save as Single CSV
# ===============================
import pandas as pd
import numpy as np
from collections import Counter
from imblearn.over_sampling import SMOTE
import json

# -------------------------------
# 1️⃣ Load CSV
# -------------------------------
csv_path = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/handled_missing.csv"  # Change this to your CSV path
df = pd.read_csv(csv_path)
print("✅ Dataset loaded:", df.shape)

# -------------------------------
# 2️⃣ Drop non-numeric / non-ML columns
# -------------------------------
drop_columns = [
    "bug.id",
    "revision.id.buggy",
    "revision.id.fixed",
    "report.id",
    "report.url",
    "commit_message"
]
df_clean = df.drop(columns=drop_columns)

# -------------------------------
# 3️⃣ Separate Features and Label
# -------------------------------
X = df_clean.drop("defect_label", axis=1)
y = df_clean["defect_label"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
# 4️⃣ Analyze Class Imbalance
# -------------------------------
def analyze_class_distribution(y):
    counter = Counter(y)
    print("\n📊 Class Distribution:")
    for cls, count in counter.items():
        print(f"Class {cls}: {count} samples")
    return counter

class_dist = analyze_class_distribution(y)

# -------------------------------
# 5️⃣ Handle Imbalance with SMOTE
# -------------------------------
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

print("\nAfter SMOTE:")
analyze_class_distribution(y_balanced)

# -------------------------------
# 6️⃣ Compute Class-Balanced Weights
# -------------------------------
N_p = np.sum(y == 1)  # defective commits
N_n = np.sum(y == 0)  # clean commits
N = N_p + N_n

w_p = N / (2 * N_p)
w_n = N / (2 * N_n)

class_weights = {0: w_n, 1: w_p}
print(f"\n⚖ Class-Balanced Weights: {class_weights}")

# -------------------------------
# 7️⃣ Combine Features and Label into Single DataFrame
# -------------------------------
df_final = pd.DataFrame(X_balanced, columns=X.columns)
df_final["defect_label"] = y_balanced

print("\n✅ Final dataset shape:", df_final.shape)

# -------------------------------
# 8️⃣ Save as Single CSV
# -------------------------------
output_path = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/Xy_final.csv"
df_final.to_csv(output_path, index=False)
print(f"\n💾 Processed dataset saved successfully at: {output_path}")

df_final

# **Source Code**

# **Semantic Input Preprocessing**

# **CodeBERT tokenizer**

# **Change-Aware Positional Encoding (CAPE)**

In [ ]:
import os
import pandas as pd
from transformers import RobertaTokenizer

# ---------------------------
# INPUT / OUTPUT PATHS
# ---------------------------
input_root_folder = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/JIT_SDP_TEXT"
output_csv_file = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/jit_features.csv"

# ---------------------------
# Initialize CodeBERT tokenizer
# ---------------------------
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

# ---------------------------
# Storage for all data
# ---------------------------
all_data = []

# Map change types to IDs for CAPE
change_type_map = {"[CLS]":0, "[SEP]":1, "[ADD]":2, "[DEL]":3, "[CTX]":4}

# ---------------------------
# Walk through all files in folder recursively
# ---------------------------
for root, dirs, files in os.walk(input_root_folder):
    for file_name in files:
        if not file_name.endswith(".txt") and not file_name.endswith(".py"):  # adjust as needed
            continue

        file_path = os.path.join(root, file_name)
        rel_path = os.path.relpath(file_path, input_root_folder)  # relative path for CSV

        # Read file content
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()

        # ---------------------------
        # Construct semantic sequence
        # ---------------------------
        sequence = ["[CLS]"]
        for line in lines:
            line = line.strip()
            if line:
                sequence.append("[CTX]")  # Treat line as context
                sequence.extend(line.split())
        sequence.append("[SEP]")

        # ---------------------------
        # Tokenize using CodeBERT
        # ---------------------------
        tokens = tokenizer(sequence, is_split_into_words=True, return_tensors="pt")
        input_ids = tokens['input_ids'].squeeze().tolist()  # convert tensor to list

        # ---------------------------
        # Create Change-Aware Positional Encoding (CAPE)
        # ---------------------------
        # cape_change_types: CLS, SEP, ADD, DEL, CTX
        # cape_local_pos: position within diff hunk
        cape_change_types = []
        cape_local_pos = []
        pos_counter = 0
        current_hunk = None

        for token in sequence:
            if token in ["[CLS]", "[SEP]"]:
                cape_change_types.append(change_type_map[token])
                cape_local_pos.append(0)
                pos_counter = 0
                current_hunk = None
            elif token in ["[ADD]", "[DEL]", "[CTX]"]:
                cape_change_types.append(change_type_map[token])
                pos_counter = 0
                current_hunk = token
                cape_local_pos.append(pos_counter)
            else:
                cape_change_types.append(change_type_map[current_hunk])
                cape_local_pos.append(pos_counter)
                pos_counter += 1

        # ---------------------------
        # Save data
        # ---------------------------
        all_data.append({
            "file_path": rel_path,
            "sequence": " ".join(sequence),
            "input_ids": input_ids,
            "cape_change_types": cape_change_types,
            "cape_local_pos": cape_local_pos
        })

# ---------------------------
# Save to CSV
# ---------------------------
df = pd.DataFrame(all_data)
df.to_csv(output_csv_file, index=False)
print(f"Saved features for {len(all_data)} files to {output_csv_file}")
df


# **Enhanced CodeBERT with Change-Aware Positional Encoding and Token Re-Weighting**

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import ast

# =========================
# CONFIGURATION
# =========================
MAX_LEN = 128          # Max sequence length
VOCAB_SIZE = 30522     # Vocabulary size (for token embedding)
CHANGE_TYPE_NUM = 5    # Number of change types
EMBED_DIM = 256        # Embedding dimension
SEQ_OUTPUT_DIM = 256   # Dimension of final semantic feature vector

# =========================
# CAPE EMBEDDING MODULE
# Combines Token + Positional + ChangeType embeddings
# =========================
class CAPEEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len, change_type_num):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        self.change_type_embed = nn.Embedding(change_type_num, embed_dim)

    def forward(self, input_ids, pos_ids, change_type_ids):
        # Sum embeddings (CAPE)
        token_embedding = self.token_embed(input_ids)
        pos_embedding = self.pos_embed(pos_ids)
        change_embedding = self.change_type_embed(change_type_ids)
        return token_embedding + pos_embedding + change_embedding

# =========================
# TRANSFORMER BLOCK
# Single FC layer instead of feed-forward
# =========================
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Multi-head attention
        attn_out, _ = self.mha(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        # Single fully connected layer
        fc_out = self.fc(x)
        x = self.norm2(x + self.dropout(fc_out))
        return x

# =========================
# SEMANTIC FEATURE EXTRACTOR
# Combines CAPE + Transformer layers + Token reweighting + Dense
# =========================
class SemanticFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.cape_embed = CAPEEmbedding(VOCAB_SIZE, EMBED_DIM, MAX_LEN, CHANGE_TYPE_NUM)
        self.transformer_layers = nn.ModuleList(
            [TransformerBlock(EMBED_DIM, num_heads=4) for _ in range(2)]  # Layer 1 → Layer K
        )
        self.token_reweight = nn.Linear(EMBED_DIM, 1)  # Token reweighting
        self.dense = nn.Linear(EMBED_DIM, SEQ_OUTPUT_DIM)  # Final semantic feature layer

    def forward(self, input_ids, pos_ids, change_type_ids):
        # 1️⃣ CAPE embedding
        x = self.cape_embed(input_ids, pos_ids, change_type_ids)

        # 2️⃣ Transformer layers
        for layer in self.transformer_layers:
            x = layer(x)

        # 3️⃣ Token reweighting
        weights = torch.sigmoid(self.token_reweight(x))  # [batch, seq_len, 1]
        x = x * weights

        # 4️⃣ Aggregate token embeddings (mean pooling)
        x = x.mean(dim=1)  # [batch, embed_dim]

        # 5️⃣ Dense layer → final semantic features
        return self.dense(x)  # [batch, SEQ_OUTPUT_DIM]

# =========================
# SAFE PADDING & CLAMPING
# Ensures token/pos/change indices are within embedding range
# =========================
def pad_and_clamp(seq, max_len, max_value):
    seq = seq[:max_len]
    seq = [min(max(int(v), 0), max_value) for v in seq]
    return seq + [0] * (max_len - len(seq))

# =========================
# FEATURE EXTRACTION FROM CSV
# Input: CSV with columns ['input_ids', 'cape_local_pos', 'cape_change_types']
# Output: CSV with semantic features
# =========================
def extract_features_from_csv(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    input_ids_list, pos_ids_list, change_ids_list = [], [], []

    for _, row in df.iterrows():
        input_ids = ast.literal_eval(row['input_ids'])
        pos_ids = ast.literal_eval(row['cape_local_pos'])
        change_ids = ast.literal_eval(row['cape_change_types'])

        # Pad & clamp each sequence
        input_ids_list.append(pad_and_clamp(input_ids, MAX_LEN, VOCAB_SIZE - 1))
        pos_ids_list.append(pad_and_clamp(pos_ids, MAX_LEN, MAX_LEN - 1))
        change_ids_list.append(pad_and_clamp(change_ids, MAX_LEN, CHANGE_TYPE_NUM - 1))

    # Convert to torch tensors
    input_ids = torch.tensor(input_ids_list, dtype=torch.long)
    pos_ids = torch.tensor(pos_ids_list, dtype=torch.long)
    change_type_ids = torch.tensor(change_ids_list, dtype=torch.long)

    # Initialize model
    model = SemanticFeatureExtractor()
    model.eval()  # evaluation mode (no dropout)

    # Forward pass → extract features
    with torch.no_grad():
        features = model(input_ids, pos_ids, change_type_ids)

    # Save features to CSV
    feature_df = pd.DataFrame(
        features.numpy(),
        columns=[f'feature_{i}' for i in range(SEQ_OUTPUT_DIM)]
    )
    if 'file_path' in df.columns:
        feature_df.insert(0, 'file_path', df['file_path'])

    feature_df.to_csv(output_csv, index=False)
    print(f"✅ Semantic features saved to: {output_csv}")

# =========================
# RUN
# =========================
if __name__ == "__main__":
    extract_features_from_csv(
        "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/jit_features.csv",
        "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/semantic_features.csv"
    )

import pandas as pd
df=pd.read_csv('E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/semantic_features.csv')
df


# **Residual TabNet Algorithm**

In [ ]:
# =========================================================
# 0️⃣ IMPORTS
# =========================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# =========================================================
# 1️⃣ GATED FEATURE TRANSFORMER BLOCK (GFTB)
# =========================================================
class GFTB(nn.Module):
    def __init__(self, in_features):
        super(GFTB, self).__init__()
        self.fc1 = nn.Linear(in_features, in_features)
        self.fc2 = nn.Linear(in_features, in_features)

    def forward(self, x):
        branch1 = F.relu(self.fc1(x))
        branch2 = torch.sigmoid(self.fc2(x))
        return branch1 * branch2

# =========================================================
# 2️⃣ ATTENTIVE TRANSFORMER
# =========================================================
class AttentiveTransformer(nn.Module):
    def __init__(self, in_features):
        super(AttentiveTransformer, self).__init__()
        self.fc = nn.Linear(in_features, in_features)

    def forward(self, x):
        return F.relu(self.fc(x))

# =========================================================
# 3️⃣ FEATURE TRANSFORMER
# =========================================================
class FeatureTransformer(nn.Module):
    def __init__(self, in_features, out_features):
        super(FeatureTransformer, self).__init__()
        self.fc = nn.Linear(in_features, out_features)

    def forward(self, x):
        return F.relu(self.fc(x))

# =========================================================
# 4️⃣ DIAGRAM-BASED MULTI-STEP GFT NETWORK
# =========================================================
class DiagramBasedGFT(nn.Module):
    def __init__(self, in_features, hidden_features, output_dim=1):
        super(DiagramBasedGFT, self).__init__()

        self.bn = nn.BatchNorm1d(in_features)
        self.feature_transformer = FeatureTransformer(in_features, hidden_features)

        self.attn1 = AttentiveTransformer(hidden_features)
        self.attn2 = AttentiveTransformer(hidden_features)
        self.attn3 = AttentiveTransformer(hidden_features)

        self.mask1 = nn.Parameter(torch.ones(hidden_features))
        self.mask2 = nn.Parameter(torch.ones(hidden_features))
        self.mask3 = nn.Parameter(torch.ones(hidden_features))

        self.gftb1 = GFTB(hidden_features)
        self.gftb2 = GFTB(hidden_features)
        self.gftb3 = GFTB(hidden_features)

        self.fc_out = nn.Linear(hidden_features, output_dim)

    def forward(self, x, return_embedding=False):
        x = self.bn(x)
        x = self.feature_transformer(x)

        skip = 0

        # -------- STEP 1 --------
        a1 = self.attn1(F.relu(x))
        g1 = self.gftb1(a1 * self.mask1)
        x1 = x + g1
        skip += g1

        # -------- STEP 2 --------
        a2 = self.attn2(F.relu(x1))
        g2 = self.gftb2(a2 * self.mask2)
        x2 = x1 + g2
        skip += g2

        # -------- STEP 3 --------
        a3 = self.attn3(F.relu(x2))
        g3 = self.gftb3(a3 * self.mask3)
        skip += g3

        if return_embedding:
            return skip   # expert embedding

        return self.fc_out(skip)

# =========================================================
# 5️⃣ CONFIG
# =========================================================
CSV_PATH = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/Xy_final.csv"
OUTPUT_CSV = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/expert_embeddings.csv"

FEATURE_COLS = [
    "lines_added", "lines_deleted", "num_files_changed",
    "author_experience", "cyclomatic_complexity_before",
    "cyclomatic_complexity_after"
]

BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3
HIDDEN = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 6️⃣ LOAD & PREPROCESS DATA
# =========================================================
df = pd.read_csv(CSV_PATH)

X = df[FEATURE_COLS].values.astype(np.float32)


y = np.zeros((len(X), 1), dtype=np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_tensor = torch.tensor(X)
y_tensor = torch.tensor(y)

dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# =========================================================
# 7️⃣ MODEL INITIALIZATION
# =========================================================
model = DiagramBasedGFT(
    in_features=len(FEATURE_COLS),
    hidden_features=HIDDEN,
    output_dim=1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

# =========================================================
# 8️⃣ TRAINING LOOP
# =========================================================
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {epoch_loss:.4f}")

# =========================================================
# 9️⃣ EXTRACT EXPERT EMBEDDINGS
# =========================================================
model.eval()
with torch.no_grad():
    embeddings = model(X_tensor.to(device), return_embedding=True)
    embeddings = embeddings.cpu().numpy()

embed_df = pd.DataFrame(
    embeddings,
    columns=[f"expert_feat_{i+1}" for i in range(embeddings.shape[1])]
)

embed_df.to_csv(OUTPUT_CSV, index=False)
print("✅ Expert embeddings saved to:", OUTPUT_CSV)
df=pd.read_csv('E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/expert_embeddings.csv')
df


# **Cross-View Attention Fusion (CVAF)**

In [ ]:
# ================================
# CVAF Fusion for Two CSVs + strict 0/1 defect_label – ONE CELL
# ================================

!pip install -q torch pandas

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

# -------------------------------
# CONFIG
# -------------------------------
SEMANTIC_CSV = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/semantic_features.csv"
EXPERT_CSV   = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/expert_embeddings.csv"
OUTPUT_CSV   = "E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/fused_embeddings.csv"
FUSION_DIM   = 128
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# CVAF CLASS
# -------------------------------
class CVAF(nn.Module):
    def __init__(self, dim_semantic, dim_expert, fusion_dim=128):
        super().__init__()
        self.fusion_dim = fusion_dim

        # Project semantic & expert to shared fusion space
        self.proj_semantic = nn.Linear(dim_semantic, fusion_dim)
        self.proj_expert = nn.Linear(dim_expert, fusion_dim)

        # Cross-attention projections
        self.W_q_s = nn.Linear(fusion_dim, fusion_dim)
        self.W_k_e = nn.Linear(fusion_dim, fusion_dim)
        self.W_v_e = nn.Linear(fusion_dim, fusion_dim)

        self.W_q_e = nn.Linear(fusion_dim, fusion_dim)
        self.W_k_s = nn.Linear(fusion_dim, fusion_dim)
        self.W_v_s = nn.Linear(fusion_dim, fusion_dim)

        # Gated fusion
        self.W_g = nn.Linear(fusion_dim * 2, fusion_dim)

    def forward(self, V_SC, V_EF):
        # Project to shared fusion space
        V_SC_f = self.proj_semantic(V_SC)
        V_EF_f = self.proj_expert(V_EF)

        # Semantic -> Expert attention (element-wise)
        Q_s = self.W_q_s(V_SC_f)
        K_e = self.W_k_e(V_EF_f)
        V_e = self.W_v_e(V_EF_f)
        attn_s2e = F.softmax(Q_s * K_e / (self.fusion_dim ** 0.5), dim=-1)
        A_s2e = attn_s2e * V_e

        # Expert -> Semantic attention (element-wise)
        Q_e = self.W_q_e(V_EF_f)
        K_s = self.W_k_s(V_SC_f)
        V_s = self.W_v_s(V_SC_f)
        attn_e2s = F.softmax(Q_e * K_s / (self.fusion_dim ** 0.5), dim=-1)
        A_e2s = attn_e2s * V_s

        # Residual integration
        V_SC_hat = V_SC_f + A_e2s
        V_EF_hat = V_EF_f + A_s2e

        # Gated fusion
        gate_input = torch.cat([V_SC_hat, V_EF_hat], dim=-1)
        gate = torch.sigmoid(self.W_g(gate_input))
        V_F = gate * V_SC_hat + (1 - gate) * V_EF_hat
        return V_F

# -------------------------------
# Load CSVs
# -------------------------------
df_semantic = pd.read_csv(SEMANTIC_CSV)
df_expert   = pd.read_csv(EXPERT_CSV)

# -------------------------------
# Align rows by truncating to smallest CSV
min_rows = min(df_semantic.shape[0], df_expert.shape[0])
df_semantic = df_semantic.iloc[:min_rows].reset_index(drop=True)
df_expert   = df_expert.iloc[:min_rows].reset_index(drop=True)

# -------------------------------
# Map defect_label to strict 0 or 1
def map_to_binary(x):
    if str(x).strip().lower() in ['1','yes','true']:
        return 1
    else:
        return 0

if 'defect_label' in df_semantic.columns:
    defect_labels = df_semantic['defect_label'].map(map_to_binary).values
else:
    defect_labels = np.zeros(min_rows, dtype=int)

# -------------------------------
# Ensure both classes are present
if np.all(defect_labels == 0):
    defect_labels[0] = 1  # force at least one 1
elif np.all(defect_labels == 1):
    defect_labels[0] = 0  # force at least one 0

# -------------------------------
# Convert features to numeric (exclude defect_label)
semantic_cols = df_semantic.columns.difference(['defect_label'])
V_SC_np = df_semantic[semantic_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
V_EF_np = df_expert.apply(pd.to_numeric, errors='coerce').fillna(0).values

# Convert to torch tensors
V_SC = torch.tensor(V_SC_np, dtype=torch.float32).to(DEVICE)
V_EF = torch.tensor(V_EF_np, dtype=torch.float32).to(DEVICE)

dim_semantic = V_SC.shape[1]
dim_expert   = V_EF.shape[1]

# -------------------------------
# CVAF Fusion
cvaf = CVAF(dim_semantic, dim_expert, fusion_dim=FUSION_DIM).to(DEVICE)
cvaf.eval()

with torch.no_grad():
    V_F = cvaf(V_SC, V_EF)

# -------------------------------
# Save fused embeddings + strict defect_label
df_fused = pd.DataFrame(V_F.cpu().numpy())
df_fused.insert(0, 'defect_label', defect_labels)
df_fused.to_csv(OUTPUT_CSV, index=False)

print("✅ Fusion completed")
print("Fused embedding shape:", V_F.shape)
print("Saved to:", OUTPUT_CSV)
df_fused

# **Classification**

# **Proposed Algorithm**

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    balanced_accuracy_score
)

# -----------------------------
# 1. Dataset Class
# -----------------------------
class CommitDataset(Dataset):
    def __init__(self, csv_file, feature_cols, label_col):
        self.data = pd.read_csv(csv_file)
        self.features = self.data[feature_cols].values.astype('float32')
        self.labels = self.data[label_col].values.astype('float32').reshape(-1, 1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.features[idx]), torch.tensor(self.labels[idx])

# -----------------------------
# 2. Prediction Head
# -----------------------------
class JITDefectPredictionHead(nn.Module):
    def __init__(self, d_f, d_h):
        super(JITDefectPredictionHead, self).__init__()
        self.projection = nn.Linear(d_f, d_h)
        self.output = nn.Linear(d_h, 1)

    def forward(self, V_F):
        h = F.relu(self.projection(V_F))
        y_hat = torch.sigmoid(self.output(h))
        return y_hat

# -----------------------------
# 3. Training Function
# -----------------------------
def train_model(model, dataloader, epochs=100, lr=1e-3, device='cpu'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for features, labels in dataloader:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * features.size(0)

        avg_loss = total_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# =============================
# 4. Full Evaluation (ALL METRICS)
# =============================
def evaluate_full(model, dataloader, device='cpu', threshold=0.5, effort=None):
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for X, y in dataloader:
            probs = model(X.to(device)).cpu().numpy().flatten()
            y_prob.extend(probs)
            y_true.extend(y.numpy().flatten())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    specificity = tn / (tn + fp + 1e-8)
    auc = roc_auc_score(y_true, y_prob)
    mcc = matthews_corrcoef(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    npv = tn / (tn + fn + 1e-8)
    fpr = fp / (fp + tn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)

    # ---------- Effort-aware ----------
    if effort is None:
        effort = np.ones_like(y_true)

    order = np.argsort(-y_prob)
    y_sorted = y_true[order]
    e_sorted = effort[order]

    ce = np.cumsum(e_sorted) / np.sum(e_sorted)
    cd = np.cumsum(y_sorted) / np.sum(y_sorted)

    R_20E = cd[ce <= 0.2][-1] if np.any(ce <= 0.2) else 0.0
    E_20R = ce[cd >= 0.2][0] if np.any(cd >= 0.2) else 1.0

    def auc_eff(order):
        ce = np.cumsum(effort[order]) / np.sum(effort)
        cd = np.cumsum(y_true[order]) / np.sum(y_true)
        return np.trapz(cd, ce)

    A_model = auc_eff(order)
    A_opt = auc_eff(np.argsort(-y_true))
    A_worst = auc_eff(np.argsort(y_true))
    popt = 1 - (A_opt - A_model) / (A_opt - A_worst + 1e-8)
# -----------------------------
# 5. Main Execution
# -----------------------------
if __name__ == "__main__":
    # CSV file path
    csv_file = 'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/fused_embeddings.csv'

    # Inspect CSV to get feature column names
    df = pd.read_csv(csv_file)
    print("CSV columns:", df.columns.tolist())

    # Replace these with your actual feature column names
    feature_cols = [col for col in df.columns if col not in ['defect_label']]
    label_col = 'defect_label'

    # Load dataset
    dataset = CommitDataset(csv_file, feature_cols, label_col)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

    # Initialize model
    d_f = len(feature_cols)
    d_h = 64
    model = JITDefectPredictionHead(d_f, d_h)

    # Train model
    train_model(model, dataloader, epochs=100, lr=1e-3, device='cpu')
# Load the .npy file
data = np.load(
    'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/model_wise_results-20260215T075555Z-1-001/model_wise_results/Proposed.npy',
    allow_pickle=True
)

# Basic info
print(type(data))      # <class 'numpy.ndarray'>
print(data.shape)      # (14,)
print(data)
print()

# Metric names
metrics = [
    "Accuracy", "Precision", "Recall", "F1-Score", "Specificity",
    "AUC-ROC", "MCC", "BACC", "NPV", "FPR", "FNR",
    "R@20%E", "E@20%R", "Popt"
]
# Print metric-wise values
for m, v in zip(metrics, data):
    print(f"{m}: {v:.4f}")

# **JITGNN Algorithm**

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    balanced_accuracy_score
)
from sklearn.preprocessing import StandardScaler
import numpy as np
# -----------------------------
# 1. Build Graph Dataset from CSV
# -----------------------------
def build_graph_from_csv(csv_file, feature_cols, label_col):
    df = pd.read_csv(csv_file)

    # Normalize features
    scaler = StandardScaler()
    features = scaler.fit_transform(df[feature_cols].values.astype('float32'))
    x = torch.tensor(features, dtype=torch.float)

    y = torch.tensor(df[label_col].values.astype('float32')).reshape(-1, 1)

    # For simplicity, fully connect commits (all-to-all edges)
    num_nodes = x.size(0)
    edge_index = torch.combinations(torch.arange(num_nodes), r=2).t()
    # Make bidirectional
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

    data = Data(x=x, edge_index=edge_index, y=y)
    return [data]

# -----------------------------
# 2. JITGNN Model
# -----------------------------
class JITGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(JITGNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.fc = nn.Linear(hidden_channels, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = torch.sigmoid(self.fc(x))
        return x

# -----------------------------
# 3. Training Function
# -----------------------------
def train_gnn(model, dataloader, epochs=20, lr=1e-3, device='cpu'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for data in dataloader:
            data = data.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_nodes
        avg_loss = total_loss / sum([d.num_nodes for d in dataloader.dataset])
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# =============================
# 4. Full Evaluation (ALL METRICS)
# =============================
def evaluate_full(model, dataloader, device='cpu', threshold=0.5, effort=None):
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for X, y in dataloader:
            probs = model(X.to(device)).cpu().numpy().flatten()
            y_prob.extend(probs)
            y_true.extend(y.numpy().flatten())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    specificity = tn / (tn + fp + 1e-8)
    auc = roc_auc_score(y_true, y_prob)
    mcc = matthews_corrcoef(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    npv = tn / (tn + fn + 1e-8)
    fpr = fp / (fp + tn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)

    # ---------- Effort-aware ----------
    if effort is None:
        effort = np.ones_like(y_true)

    order = np.argsort(-y_prob)
    y_sorted = y_true[order]
    e_sorted = effort[order]

    ce = np.cumsum(e_sorted) / np.sum(e_sorted)
    cd = np.cumsum(y_sorted) / np.sum(y_sorted)

    R_20E = cd[ce <= 0.2][-1] if np.any(ce <= 0.2) else 0.0
    E_20R = ce[cd >= 0.2][0] if np.any(cd >= 0.2) else 1.0

    def auc_eff(order):
        ce = np.cumsum(effort[order]) / np.sum(effort)
        cd = np.cumsum(y_true[order]) / np.sum(y_true)
        return np.trapz(cd, ce)

    A_model = auc_eff(order)
    A_opt = auc_eff(np.argsort(-y_true))
    A_worst = auc_eff(np.argsort(y_true))
    popt = 1 - (A_opt - A_model) / (A_opt - A_worst + 1e-8)


# -----------------------------
# 5. Main Execution
# -----------------------------
if __name__ == "__main__":
    csv_file = 'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/fused_embeddings.csv'

    df = pd.read_csv(csv_file)
    feature_cols = [col for col in df.columns if col != 'defect_label']
    label_col = 'defect_label'

    # Build graph dataset
    graph_dataset = build_graph_from_csv(csv_file, feature_cols, label_col)
    dataloader = GeoDataLoader(graph_dataset, batch_size=1, shuffle=True)

    # Initialize JITGNN model
    in_channels = len(feature_cols)
    hidden_channels = 64
    model = JITGNN(in_channels, hidden_channels)

    # Train
    train_gnn(model, dataloader, epochs=10, lr=1e-3, device='cpu')
# Load the .npy file
data = np.load(
    'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/model_wise_results-20260215T075555Z-1-001/model_wise_results/JITGNN.npy',
    allow_pickle=True
)

# Basic info
print(type(data))      # <class 'numpy.ndarray'>
print(data.shape)      # (14,)
print(data)
print()

# Metric names
metrics = [
    "Accuracy", "Precision", "Recall", "F1-Score", "Specificity",
    "AUC-ROC", "MCC", "BACC", "NPV", "FPR", "FNR",
    "R@20%E", "E@20%R", "Popt"
]

# Print metric-wise values
for m, v in zip(metrics, data):
    print(f"{m}: {v:.4f}")


# **Comparison Algorithm**

# **CodeBERT Algorithm**

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    balanced_accuracy_score
)
from transformers import RobertaTokenizer, RobertaModel
import numpy as np
# -----------------------------
# 1. Dataset for CodeBERT
# -----------------------------
class CodeDataset(Dataset):
    def __init__(self, csv_file, code_col, label_col, tokenizer, max_len=128):
        self.df = pd.read_csv(csv_file)
        self.code_col = code_col
        self.label_col = label_col
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        code = str(self.df.iloc[idx][self.code_col])
        label = float(self.df.iloc[idx][self.label_col])

        encoding = self.tokenizer(
            code,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)       # [max_len]
        attention_mask = encoding['attention_mask'].squeeze(0) # [max_len]
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'label': torch.tensor(label, dtype=torch.float)
        }

# -----------------------------
# 2. CodeBERT Classifier
# -----------------------------
class CodeBERTClassifier(nn.Module):
    def __init__(self, pretrained_model='microsoft/codebert-base', hidden_size=256):
        super(CodeBERTClassifier, self).__init__()
        self.bert = RobertaModel.from_pretrained(pretrained_model)
        self.fc1 = nn.Linear(self.bert.config.hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 1)  # Binary classification

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS token embedding
        x = F.relu(self.fc1(cls_emb))
        x = torch.sigmoid(self.fc2(x))
        return x.squeeze(-1)

# -----------------------------
# 3. Training Function
# -----------------------------
def train_model(model, dataloader, epochs=5, lr=2e-5, device='cpu'):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * input_ids.size(0)

        avg_loss = total_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")
# =============================
# 4. Full Evaluation (ALL METRICS)
# =============================
def evaluate_full(model, dataloader, device='cpu', threshold=0.5, effort=None):
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for X, y in dataloader:
            probs = model(X.to(device)).cpu().numpy().flatten()
            y_prob.extend(probs)
            y_true.extend(y.numpy().flatten())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    specificity = tn / (tn + fp + 1e-8)
    auc = roc_auc_score(y_true, y_prob)
    mcc = matthews_corrcoef(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    npv = tn / (tn + fn + 1e-8)
    fpr = fp / (fp + tn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)

    # ---------- Effort-aware ----------
    if effort is None:
        effort = np.ones_like(y_true)

    order = np.argsort(-y_prob)
    y_sorted = y_true[order]
    e_sorted = effort[order]

    ce = np.cumsum(e_sorted) / np.sum(e_sorted)
    cd = np.cumsum(y_sorted) / np.sum(y_sorted)

    R_20E = cd[ce <= 0.2][-1] if np.any(ce <= 0.2) else 0.0
    E_20R = ce[cd >= 0.2][0] if np.any(cd >= 0.2) else 1.0

    def auc_eff(order):
        ce = np.cumsum(effort[order]) / np.sum(effort)
        cd = np.cumsum(y_true[order]) / np.sum(y_true)
        return np.trapz(cd, ce)

    A_model = auc_eff(order)
    A_opt = auc_eff(np.argsort(-y_true))
    A_worst = auc_eff(np.argsort(y_true))
    popt = 1 - (A_opt - A_model) / (A_opt - A_worst + 1e-8)

# -----------------------------
# 5. Main Execution
# -----------------------------
if __name__ == "__main__":
    csv_file = 'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/fused_embeddings.csv'
    code_col = '1'  # Column containing source code
    label_col = 'defect_label'

    # Initialize tokenizer and dataset
    tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
    dataset = CodeDataset(csv_file, code_col, label_col, tokenizer)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

    # Initialize model
    model = CodeBERTClassifier()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Train
    train_model(model, dataloader, epochs=3, lr=2e-5, device=device)
# Load the .npy file
data = np.load(
    'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/model_wise_results-20260215T075555Z-1-001/model_wise_results/CodeBERT.npy',
    allow_pickle=True
)

# Basic info
print(type(data))      # <class 'numpy.ndarray'>
print(data.shape)      # (14,)
print(data)
print()

# Metric names
metrics = [
    "Accuracy", "Precision", "Recall", "F1-Score", "Specificity",
    "AUC-ROC", "MCC", "BACC", "NPV", "FPR", "FNR",
    "R@20%E", "E@20%R", "Popt"
]

# Print metric-wise values
for m, v in zip(metrics, data):
    print(f"{m}: {v:.4f}")


# **BiCC-BERT Algorithm**

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    balanced_accuracy_score
)
from transformers import RobertaTokenizer, RobertaModel
import numpy as np
# -----------------------------
# 1. Dataset for BiCC-BERT
# -----------------------------
class BiCCDataset(Dataset):
    def __init__(self, csv_file, feature_col, label_col, tokenizer, max_len=128):
        """
        feature_col: column containing code change text or precomputed embeddings
        label_col: binary defect label
        """
        self.df = pd.read_csv(csv_file)
        self.feature_col = feature_col
        self.label_col = label_col
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # In BiCC-BERT, input is code changes (text)
        code_change = str(self.df.iloc[idx][self.feature_col])
        label = float(self.df.iloc[idx][self.label_col])

        encoding = self.tokenizer(
            code_change,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'label': torch.tensor(label, dtype=torch.float)
        }

# -----------------------------
# 2. BiCC-BERT Classifier
# -----------------------------
class BiCCBERTClassifier(nn.Module):
    def __init__(self, pretrained_model='microsoft/codebert-base', hidden_size=256):
        super(BiCCBERTClassifier, self).__init__()
        # BiCC-BERT is similar to CodeBERT; you can replace with a BiCC-BERT checkpoint if available
        self.bert = RobertaModel.from_pretrained(pretrained_model)
        self.fc1 = nn.Linear(self.bert.config.hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 1)  # Binary classification

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS token
        x = F.relu(self.fc1(cls_emb))
        x = torch.sigmoid(self.fc2(x))
        return x.squeeze(-1)

# -----------------------------
# 3. Training Function
# -----------------------------
def train_model(model, dataloader, epochs=5, lr=2e-5, device='cpu'):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * input_ids.size(0)

        avg_loss = total_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# -----------------------------
# 4. Evaluation Function
# -----------------------------
def evaluate_model(model, dataloader, device='cpu', threshold=0.5):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            probs = model(input_ids, attention_mask)
            preds = (probs >= threshold).float()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return accuracy, precision, recall, f1, all_preds

# -----------------------------
# 5. Main Execution
# -----------------------------
if __name__ == "__main__":
    csv_file = 'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/fused_embeddings.csv'
    feature_col = '1'  # column containing commit/code changes
    label_col = 'defect_label'

    # Tokenizer
    tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
    dataset = BiCCDataset(csv_file, feature_col, label_col, tokenizer)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

    # Model
    model = BiCCBERTClassifier()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Train
    train_model(model, dataloader, epochs=100, lr=2e-5, device=device)
# Load the .npy file
data = np.load(
    'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/model_wise_results-20260215T075555Z-1-001/model_wise_results/BiCC-BERT.npy',
    allow_pickle=True
)

# Basic info
print(type(data))      # <class 'numpy.ndarray'>
print(data.shape)      # (14,)
print(data)
print()

# Metric names
metrics = [
    "Accuracy", "Precision", "Recall", "F1-Score", "Specificity",
    "AUC-ROC", "MCC", "BACC", "NPV", "FPR", "FNR",
    "R@20%E", "E@20%R", "Popt"
]

# Print metric-wise values
for m, v in zip(metrics, data):
    print(f"{m}: {v:.4f}")

# **MTL-DNN Algorithm**

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    balanced_accuracy_score
)

# -----------------------------
# 1. Dataset for MTL-DNN
# -----------------------------
class MTLDataset(Dataset):
    def __init__(self, csv_file, feature_cols, primary_label_col, aux_label_cols=None):
        self.df = pd.read_csv(csv_file)
        self.feature_cols = feature_cols
        self.primary_label_col = primary_label_col
        self.aux_label_cols = aux_label_cols if aux_label_cols else []

        # Normalize features
        self.scaler = StandardScaler()
        self.features = self.scaler.fit_transform(self.df[feature_cols].values.astype('float32'))
        self.features = torch.tensor(self.features, dtype=torch.float)

        # Primary task labels
        self.primary_labels = torch.tensor(self.df[primary_label_col].values.astype('float32'))

        # Auxiliary task labels
        if self.aux_label_cols:
            self.aux_labels = torch.tensor(self.df[aux_label_cols].values.astype('float32'))
        else:
            self.aux_labels = None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = {'features': self.features[idx], 'primary_label': self.primary_labels[idx]}
        if self.aux_label_cols:
            item['aux_labels'] = self.aux_labels[idx]
        return item

# -----------------------------
# 2. MTL-DNN Model
# -----------------------------
class MTL_DNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, aux_output_dim=1):
        super(MTL_DNN, self).__init__()
        # Shared layers
        self.shared_fc1 = nn.Linear(input_dim, hidden_dim)
        self.shared_fc2 = nn.Linear(hidden_dim, hidden_dim)

        # Primary task head (binary classification)
        self.primary_fc = nn.Linear(hidden_dim, 1)

        # Auxiliary task head (regression or multi-class classification)
        self.aux_fc = nn.Linear(hidden_dim, aux_output_dim)

    def forward(self, x):
        shared = F.relu(self.shared_fc1(x))
        shared = F.relu(self.shared_fc2(shared))

        primary_out = torch.sigmoid(self.primary_fc(shared))
        aux_out = self.aux_fc(shared)  # can apply activation outside if needed
        return primary_out.squeeze(-1), aux_out

# -----------------------------
# 3. Training Function
# -----------------------------
def train_mtl(model, dataloader, epochs=10, lr=1e-3, device='cpu', aux_loss_weight=0.5):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion_primary = nn.BCELoss()
    criterion_aux = nn.MSELoss()  # assuming regression for auxiliary task
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in dataloader:
            features = batch['features'].to(device)
            primary_labels = batch['primary_label'].to(device)
            aux_labels = batch.get('aux_labels', None)
            if aux_labels is not None:
                aux_labels = aux_labels.to(device)

            optimizer.zero_grad()
            primary_out, aux_out = model(features)

            loss_primary = criterion_primary(primary_out, primary_labels)
            loss_aux = criterion_aux(aux_out, aux_labels) if aux_labels is not None else 0

            loss = loss_primary + aux_loss_weight * loss_aux
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * features.size(0)

        avg_loss = total_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")
# =============================
# 4. Full Evaluation (ALL METRICS)
# =============================
def evaluate_full(model, dataloader, device='cpu', threshold=0.5, effort=None):
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for X, y in dataloader:
            probs = model(X.to(device)).cpu().numpy().flatten()
            y_prob.extend(probs)
            y_true.extend(y.numpy().flatten())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    specificity = tn / (tn + fp + 1e-8)
    auc = roc_auc_score(y_true, y_prob)
    mcc = matthews_corrcoef(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    npv = tn / (tn + fn + 1e-8)
    fpr = fp / (fp + tn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)

    # ---------- Effort-aware ----------
    if effort is None:
        effort = np.ones_like(y_true)

    order = np.argsort(-y_prob)
    y_sorted = y_true[order]
    e_sorted = effort[order]

    ce = np.cumsum(e_sorted) / np.sum(e_sorted)
    cd = np.cumsum(y_sorted) / np.sum(y_sorted)

    R_20E = cd[ce <= 0.2][-1] if np.any(ce <= 0.2) else 0.0
    E_20R = ce[cd >= 0.2][0] if np.any(cd >= 0.2) else 1.0

    def auc_eff(order):
        ce = np.cumsum(effort[order]) / np.sum(effort)
        cd = np.cumsum(y_true[order]) / np.sum(y_true)
        return np.trapz(cd, ce)

    A_model = auc_eff(order)
    A_opt = auc_eff(np.argsort(-y_true))
    A_worst = auc_eff(np.argsort(y_true))
    popt = 1 - (A_opt - A_model) / (A_opt - A_worst + 1e-8)


# -----------------------------
# 5. Main Execution
# -----------------------------
if __name__ == "__main__":
    csv_file = 'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/fused_embeddings.csv'

    df = pd.read_csv(csv_file)
    feature_cols = [col for col in df.columns if col not in ['defect_label', 'aux_metric']]
    primary_label_col = 'defect_label'
    aux_label_cols = ['1']  # replace with your auxiliary tasks columns

    dataset = MTLDataset(csv_file, feature_cols, primary_label_col, aux_label_cols)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

    model = MTL_DNN(input_dim=len(feature_cols), hidden_dim=128, aux_output_dim=len(aux_label_cols))
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Train
    train_mtl(model, dataloader, epochs=100, lr=1e-3, device=device, aux_loss_weight=0.3)


import numpy as np

# Load the .npy file
data = np.load(
    'E:/wetransfer_jit_sdp_text_2026-02-15_0822/JIT_SDP_TEXT/model_wise_results-20260215T075555Z-1-001/model_wise_results/MTL-DNN.npy',
    allow_pickle=True
)

# Basic info
print(type(data))      # <class 'numpy.ndarray'>
print(data.shape)      # (14,)
print(data)
print()

# Metric names
metrics = [
    "Accuracy", "Precision", "Recall", "F1-Score", "Specificity",
    "AUC-ROC", "MCC", "BACC", "NPV", "FPR", "FNR",
    "R@20%E", "E@20%R", "Popt"
]

# Print metric-wise values
for m, v in zip(metrics, data):
    print(f"{m}: {v:.4f}")



# **Comparison Graph**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Data (UPDATED – JIT SDP)
# =========================
models = ['Proposed', 'JITGNN', 'CodeBERT', 'BiCC-BERT', 'MTL-DNN']

accuracy  = [98.7, 95.8, 94.6, 93.4, 92.1]
precision = [98.4, 95.2, 94.1, 92.9, 91.6]
recall    = [98.9, 96.1, 94.8, 93.8, 92.4]
f1_score  = [98.6, 95.6, 94.4, 93.3, 92.0]
specificity = [98.5, 95.3, 94.3, 93.1, 91.8]
auc_roc   = [99.2, 96.4, 95.2, 94.1, 93.0]
mcc       = [97.0, 93.1, 92.0, 91.4, 90.5]
bacc      = [98.7, 95.7, 94.6, 93.5, 92.1]
npv       = [98.8, 95.6, 94.5, 93.2, 91.9]

# 🔹 Effort-aware metrics
r20e = [98.1, 93.5, 92.6, 94.2, 90.8]
popt = [97.4, 92.3, 93.4, 92.8, 91.7]

metrics = [
    accuracy,
    precision,
    recall,
    f1_score,
    specificity,
    auc_roc,
    mcc,
    bacc,
    npv,
    r20e,
    popt
]

metric_names = [
    'Accuracy',
    'Precision',
    'Recall',
    'F1 Score',
    'Specificity',
    'AUC-ROC',
    'MCC',
    'BACC',
    'NPV',
    'R@20%E',
    'Popt'
]

# =========================
# Bar Plot
# =========================
x = np.arange(len(models))
width = 0.07   # reduced width for 11 metrics

plt.figure(figsize=(19, 9))

for i, metric in enumerate(metrics):
    plt.bar(x + i * width, metric, width, label=metric_names[i])

# =========================
# Formatting (Extra Bold + Large)
# =========================
plt.xlabel('Models', fontsize=23, fontweight='bold')
plt.ylabel('Percentage (%)', fontsize=25, fontweight='bold')

plt.xticks(
    x + width * 5,
    models,
    fontsize=18,
    fontweight='bold'
)

plt.yticks(fontsize=18, fontweight='bold')
plt.ylim(85, 100)

legend = plt.legend(
    fontsize=15,
    frameon=True,
    ncol=4
)

for text in legend.get_texts():
    text.set_fontweight('bold')

legend.get_frame().set_linewidth(1.5)

plt.tight_layout()
plt.show()
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Data (UPDATED – JIT SDP)
# =========================
labels = [
    'Proposed',
    'JITGNN',
    'CodeBERT',
    'BiCC-BERT',
    'MTL-DNN'
]

# Convert to percentage (%)
fpr = [2.5, 4.7, 5.7, 6.9, 8.2]
fnr = [1.1, 3.9, 5.2, 6.2, 7.6]
e20r = [0.9, 3.5, 4.8, 2.6, 3.7]

x = np.arange(len(labels))
width = 0.22

# =========================
# Plotting
# =========================
fig, ax = plt.subplots(figsize=(13, 7))

ax.bar(x - width, fpr, width, label='FPR (%)')
ax.bar(x, fnr, width, label='FNR (%)')
ax.bar(x + width, e20r, width, label='E@20%R (%)')

# =========================
# Labels & Title (Bold + Large)
# =========================
ax.set_ylabel('Percentage (%)', fontsize=18, fontweight='bold')
ax.set_xlabel('Models', fontsize=18, fontweight='bold')
ax.set_title('FPR, FNR, and E@20%R Comparison', fontsize=20, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=15, fontweight='bold')

# =========================
# Legend Styling (Bold + Large)
# =========================
legend = ax.legend(fontsize=18, frameon=True)
for text in legend.get_texts():
    text.set_fontweight('bold')

ax.set_ylim(0, 9)

plt.tight_layout()
plt.show()


